# `ReduCNN` Research Suite: Cats vs. Dogs 🐱🐶

This notebook executes structural pruning experiments on the **Cats vs. Dogs** dataset. This dataset provides a higher-resolution challenge (128x128) compared to CIFAR-10, testing the framework's ability to handle larger activation maps and binary classification tasks.

In [16]:
# --- STEP 0: BOOTLOADER ---
import sys, os
if os.path.exists('src'): sys.path.insert(0, os.path.abspath('src'))

try:
    import reducnn
    print(f"✅ ReduCNN loaded from: {reducnn.__file__}")
except ImportError:
    print("❌ ReduCNN not found in path. Please ensure you are in the project root.")

✅ ReduCNN loaded from: /content/drive/Othercomputers/My PC/activation-based-pruning/src/reducnn/__init__.py


In [17]:
import torch, torchvision, os, glob
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image

# 1. Custom Dataset for Kaggle's 'cat.N.jpg' structure
class KaggleCatDogDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.file_list = glob.glob(os.path.join(root_dir, '*.jpg'))
        self.classes = ['cat', 'dog']
        
    def __len__(self):
        return len(self.file_list)
        
    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        image = Image.open(img_path).convert('RGB')
        filename = os.path.basename(img_path)
        # Label is 'cat' or 'dog' based on prefix before first '.'
        label_str = filename.split('.')[0].lower()
        label = 0 if 'cat' in label_str else 1
        if self.transform: image = self.transform(image)
        return image, label

# 2. Define transforms and Load Data
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

data_dir = './data/cats_dogs/train' # Where Kaggle train.zip was extracted

if not os.path.exists(data_dir) or len(glob.glob(os.path.join(data_dir, '*.jpg'))) == 0:
    print(f"⚠️ Kaggle dataset not found at {data_dir}. Using mock data for demonstration.")
    class MockDataset(Dataset):
        def __init__(self, transform=None): self.transform = transform; self.classes=['cat','dog']
        def __len__(self): return 100
        def __getitem__(self, idx): return torch.randn(3, 128, 128), 0
    full_dataset = MockDataset(transform=transform)
else:
    full_dataset = KaggleCatDogDataset(data_dir, transform=transform)
    print(f"✅ Successfully loaded {len(full_dataset)} images from Kaggle structure.")

# 3. Split and Create Generalized Loaders
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32)

print(f"✅ Generalized Loaders ready. Classes: {full_dataset.classes}")

ModuleNotFoundError: No module named 'cat_dog_utils'

In [ ]:
from reducnn.backends.torch_backend import PyTorchAdapter
from reducnn.pruner import ReduCNNPruner

adapter = PyTorchAdapter(config={'lr': 1e-4, 'input_shape': (3, 128, 128), 'num_classes': 2})
model = adapter.get_model("vgg16")

print("🔥 Training VGG16 Baseline on Cats vs Dogs...")
adapter.train(model, train_loader, epochs=5, name="VGG16_CatDog", val_loader=test_loader)
b_acc = adapter.evaluate(model, test_loader)
print(f"\n✅ Baseline: {b_acc:.2f}%")

In [ ]:
print("
🔬 PRUNING: L1-Norm (Local 30%)")
surgeon = ReduCNNPruner(method='l1_norm', scope='local')
pruned_model, masks, duration = surgeon.prune(model, train_loader, ratio=0.3)

print("💊 Fine-tuning pruned model...")
adapter.train(pruned_model, train_loader, epochs=3, name="Heal_L1", val_loader=test_loader)
p_acc = adapter.evaluate(pruned_model, test_loader)
print(f"\n✅ Pruned Accuracy: {p_acc:.2f}%")

### C.1 PyTorch ResNet-18: Full Research Workflow (Cats vs Dogs)


In [ ]:
print('🧪 PYTORCH: ResNet-18 Full Suite (Cats vs Dogs)')
t_res_adapter = PyTorchAdapter(config={'lr': 1e-4, 'input_shape': (3, 128, 128), 'num_classes': 2})
t_res_model = t_res_adapter.get_model('resnet18')

print('1. Establishing ResNet Baseline...')
t_res_adapter.train(t_res_model, train_loader, epochs=2, name='ResNet_Base', val_loader=test_loader)
res_base_acc = t_res_adapter.evaluate(t_res_model, test_loader)

print('\n2. Performing Structural Surgery (Local 20%)...')
res_surgeon = ReduCNNPruner(method='l1_norm', scope='local')
pruned_res, res_masks, _ = res_surgeon.prune(t_res_model, train_loader, ratio=0.2)

print('\n3. Healing Phase (Fine-tuning)...')
t_res_adapter.train(pruned_res, train_loader, epochs=3, name='ResNet_Heal', val_loader=test_loader)
res_pruned_acc = t_res_adapter.evaluate(pruned_res, test_loader)

print(f'\n✅ ResNet-18 Results ({dataset_name}):')
print(f'   Baseline Acc: {res_base_acc:.2f}%')
print(f'   Pruned Acc:   {res_pruned_acc:.2f}%')
viz.plot_layer_sensitivity(res_masks, f'ResNet-18 Pruning Sensitivity ({dataset_name})')

# Part D: Performance & Inference Diagnostics
This section compares the **real-world inference speed** and **prediction quality** of the Original vs. Pruned models.

In [ ]:
import time
import torch

def benchmark_inference(model, loader, device, iterations=100):
    model.eval()
    # Extract a single sample for latency test
    it = iter(loader)
    x, _ = next(it)
    x = x[:1].to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(20): _ = model(x)
    
    # Timing
    start = time.time()
    with torch.no_grad():
        for _ in range(iterations):
            _ = model(x)
    end = time.time()
    
    latency = (end - start) / iterations * 1000 # ms
    return latency

print('⏱️ Benchmarking Latency (Batch Size = 1)...')
t_orig = benchmark_inference(t_model_resnet, test_loader, adapter.device)
t_pruned = benchmark_inference(pruned_res, test_loader, adapter.device)

print(f'   Original ResNet Latency: {t_orig:.3f} ms')
print(f'   Pruned ResNet Latency:   {t_pruned:.3f} ms')
print(f'   🚀 Speedup: {(t_orig/t_pruned):.2f}x')

In [ ]:
print('🖼️ Generating Inference Gallery...')
it = iter(test_loader)
images, labels = next(it)
images_sub = images[:8]
labels_sub = labels[:8]

with torch.no_grad():
    p_orig = torch.argmax(t_model_resnet(images_sub.to(adapter.device)), dim=1).cpu().numpy()
    p_pruned = torch.argmax(pruned_res(images_sub.to(adapter.device)), dim=1).cpu().numpy()

class_names = ['cat', 'dog']
viz.plot_inference_gallery(images_sub.numpy(), labels_sub.numpy(), 
                           p_orig, p_pruned, 
                           class_names=class_names, title='ResNet-18: Original vs. Pruned Predictions')